# 07 Submission — Generate Final CSV

Combines men's and women's ensemble predictions into the single submission CSV required by Kaggle.

**Inputs**:
- From `06_model_eval/mens/`: `ensemble_stage1_predictions.parquet`, `ensemble_stage2_predictions.parquet`
- From `06_model_eval/womens/`: `ensemble_stage1_predictions.parquet`, `ensemble_stage2_predictions.parquet`
- From `00_data_collection/`: `SampleSubmissionStage1.csv`, `SampleSubmissionStage2.csv`

**Outputs** (to S3 `07_submission/`):
- `submission_stage1.csv` — Stage 1 submission (2022–2025, for validation)
- `submission_stage2.csv` — Stage 2 submission (2026, the real submission)

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

#### Functions

In [2]:
def read_parquet_eval(gender, filename):
    """Read parquet from 06_model_eval on S3 or local."""
    try:
        return pd.read_parquet(f"s3://{BUCKET}/06_model_eval/{gender}/{filename}")
    except Exception:
        return pd.read_parquet(f"../06_model_eval/output/{filename}")

def read_csv(filename):
    """Read CSV from S3 raw data or local."""
    try:
        return pd.read_csv(f"s3://{BUCKET}/00_data_collection/{filename}")
    except Exception:
        return pd.read_csv(f"../00_data_collection/{filename}")

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
OUTPUT_PREFIX = f"s3://{BUCKET}/07_submission/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Sample Submissions and Men's Predictions

In [5]:
# Sample submissions define the required rows
sample_stage1 = read_csv("SampleSubmissionStage1.csv")
sample_stage2 = read_csv("SampleSubmissionStage2.csv")

print(f"Stage 1 sample submission: {sample_stage1.shape}")
print(f"Stage 2 sample submission: {sample_stage2.shape}")

# Men's ensemble predictions
mens_stage1 = read_parquet_eval("mens", "ensemble_stage1_predictions.parquet")
mens_stage2 = read_parquet_eval("mens", "ensemble_stage2_predictions.parquet")

print(f"\nMen's Stage 1 predictions: {mens_stage1.shape}")
print(f"Men's Stage 2 predictions: {mens_stage2.shape}")

Stage 1 sample submission: (519144, 2)
Stage 2 sample submission: (132133, 2)

Men's Stage 1 predictions: (261013, 6)
Men's Stage 2 predictions: (66430, 5)


## 2. Load Women's Predictions

In [6]:
# Women's ensemble predictions
womens_stage1 = read_parquet_eval("womens", "ensemble_stage1_predictions.parquet")
womens_stage2 = read_parquet_eval("womens", "ensemble_stage2_predictions.parquet")

print(f"Women's Stage 1 predictions: {womens_stage1.shape}")
print(f"Women's Stage 2 predictions: {womens_stage2.shape}")

Women's Stage 1 predictions: (258131, 6)
Women's Stage 2 predictions: (65703, 5)


## 3. Combine Men's and Women's Predictions

In [7]:
# Prepare predictions as ID, Pred
mens_s1 = mens_stage1[['ID', 'Pred']].copy()
mens_s2 = mens_stage2[['ID', 'Pred']].copy()
womens_s1 = womens_stage1[['ID', 'Pred']].copy()
womens_s2 = womens_stage2[['ID', 'Pred']].copy()

# Concatenate
combined_stage1 = pd.concat([mens_s1, womens_s1], ignore_index=True)
combined_stage2 = pd.concat([mens_s2, womens_s2], ignore_index=True)

print(f"Combined Stage 1: {combined_stage1.shape}")
print(f"Combined Stage 2: {combined_stage2.shape}")

Combined Stage 1: (519144, 2)
Combined Stage 2: (132133, 2)


## 4. Merge with Sample Submission

Ensure every row in the sample submission has a prediction. Fill any missing predictions with 0.5.

In [8]:
def build_submission(sample_sub, predictions):
    """Merge predictions into sample submission format."""
    sub = sample_sub[['ID']].merge(predictions[['ID', 'Pred']], on='ID', how='left')
    
    # Fill any missing with 0.5
    missing = sub['Pred'].isna().sum()
    if missing > 0:
        print(f"  Warning: {missing} rows missing predictions, filling with 0.5")
        sub['Pred'] = sub['Pred'].fillna(0.5)
    
    # Final clip
    sub['Pred'] = sub['Pred'].clip(0.02, 0.98)
    
    return sub

submission_stage1 = build_submission(sample_stage1, combined_stage1)
submission_stage2 = build_submission(sample_stage2, combined_stage2)

print(f"\nStage 1 submission: {submission_stage1.shape}")
print(f"  Pred range: [{submission_stage1['Pred'].min():.3f}, {submission_stage1['Pred'].max():.3f}]")
print(f"  Pred mean: {submission_stage1['Pred'].mean():.3f}")

print(f"\nStage 2 submission: {submission_stage2.shape}")
print(f"  Pred range: [{submission_stage2['Pred'].min():.3f}, {submission_stage2['Pred'].max():.3f}]")
print(f"  Pred mean: {submission_stage2['Pred'].mean():.3f}")


Stage 1 submission: (519144, 2)
  Pred range: [0.020, 0.980]
  Pred mean: 0.476

Stage 2 submission: (132133, 2)
  Pred range: [0.020, 0.980]
  Pred mean: 0.470


## 5. Validate Submission Format

In [9]:
def validate_submission(sub, sample, stage_name):
    """Check submission matches expected format."""
    checks = []
    
    # Row count
    match_rows = len(sub) == len(sample)
    checks.append(f"Row count: {len(sub)} (expected {len(sample)}) {'PASS' if match_rows else 'FAIL'}")
    
    # Columns
    has_cols = list(sub.columns) == ['ID', 'Pred']
    checks.append(f"Columns: {list(sub.columns)} {'PASS' if has_cols else 'FAIL'}")
    
    # No nulls
    no_nulls = sub['Pred'].isna().sum() == 0
    checks.append(f"No nulls: {'PASS' if no_nulls else 'FAIL'}")
    
    # Prediction range
    in_range = (sub['Pred'] >= 0).all() and (sub['Pred'] <= 1).all()
    checks.append(f"Pred range [0,1]: {'PASS' if in_range else 'FAIL'}")
    
    # IDs match
    ids_match = set(sub['ID']) == set(sample['ID'])
    checks.append(f"IDs match sample: {'PASS' if ids_match else 'FAIL'}")
    
    print(f"\n{stage_name} Validation:")
    for check in checks:
        print(f"  {check}")
    
    return all([match_rows, has_cols, no_nulls, in_range, ids_match])

s1_valid = validate_submission(submission_stage1, sample_stage1, "Stage 1")
s2_valid = validate_submission(submission_stage2, sample_stage2, "Stage 2")

print(f"\nAll validations passed: {s1_valid and s2_valid}")


Stage 1 Validation:
  Row count: 519144 (expected 519144) PASS
  Columns: ['ID', 'Pred'] PASS
  No nulls: PASS
  Pred range [0,1]: PASS
  IDs match sample: PASS

Stage 2 Validation:
  Row count: 132133 (expected 132133) PASS
  Columns: ['ID', 'Pred'] PASS
  No nulls: PASS
  Pred range [0,1]: PASS
  IDs match sample: PASS

All validations passed: True


#### Preview

In [10]:
# submission_stage1
submission_stage1

,ID,Pred
0,2022_1101_1102,0.780558
1,2022_1101_1103,0.451491
2,2022_1101_1104,0.199330
3,2022_1101_1105,0.846196
4,2022_1101_1106,0.850221
...,...,...
519139,2025_3477_3479,0.725157
519140,2025_3477_3480,0.429009
519141,2025_3478_3479,0.545427
519142,2025_3478_3480,0.301717


In [11]:
# submission_stage2
submission_stage2

,ID,Pred
0,2026_1101_1102,0.701437
1,2026_1101_1103,0.071462
2,2026_1101_1104,0.049278
3,2026_1101_1105,0.536046
4,2026_1101_1106,0.612158
...,...,...
132128,2026_3478_3480,0.297060
132129,2026_3478_3481,0.485691
132130,2026_3479_3480,0.302227
132131,2026_3479_3481,0.473418


## 6. Save Submissions

In [12]:
# Save to S3
for name, df in [('submission_stage1.csv', submission_stage1), ('submission_stage2.csv', submission_stage2)]:
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}"
        df.to_csv(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/07_submission/submission_stage1.csv ((519144, 2))
Saved to S3: s3://march-machine-learning-mania-2026/07_submission/submission_stage2.csv ((132133, 2))


## 6b. Save Predictions with Team Names

Save `predictions_2026.parquet` to S3 with team names mapped from the ID and a `Gender` column.

In [13]:
# Load team name lookups
mteams = read_csv("MTeams.csv")[['TeamID', 'TeamName']]
wteams = read_csv("WTeams.csv")[['TeamID', 'TeamName']]
team_lookup = pd.concat([mteams, wteams], ignore_index=True).set_index('TeamID')['TeamName'].to_dict()

# Start from the Stage 2 submission (2026 predictions)
preds_2026 = submission_stage2.copy()

# Parse ID components: 2026_XXXX_YYYY
preds_2026[['Season', 'TeamID_1', 'TeamID_2']] = preds_2026['ID'].str.split('_', expand=True).astype(int)

# Map team names
preds_2026['TeamName_1'] = preds_2026['TeamID_1'].map(team_lookup)
preds_2026['TeamName_2'] = preds_2026['TeamID_2'].map(team_lookup)

# Tag gender based on TeamID range (1000-1999 = mens, 3000-3999 = womens)
preds_2026['Gender'] = np.where(preds_2026['TeamID_1'] < 2000, 'mens', 'womens')

# Reorder columns
preds_2026 = preds_2026[['ID', 'Season', 'TeamID_1', 'TeamName_1', 'TeamID_2', 'TeamName_2', 'Pred', 'Gender']]

print(f"predictions_2026 shape: {preds_2026.shape}")
print(f"\nGender counts:\n{preds_2026['Gender'].value_counts()}")
print(f"\nSample rows:")
preds_2026.head(3)

predictions_2026 shape: (132133, 8)

Gender counts:
Gender
mens      66430
womens    65703
Name: count, dtype: int64

Sample rows:


,ID,Season,TeamID_1,TeamName_1,TeamID_2,TeamName_2,Pred,Gender
0,2026_1101_1102,2026,1101,Abilene Chr,1102,Air Force,0.701437,mens
1,2026_1101_1103,2026,1101,Abilene Chr,1103,Akron,0.071462,mens
2,2026_1101_1104,2026,1101,Abilene Chr,1104,Alabama,0.049278,mens


In [14]:
# Save to S3
s3_path = f"{OUTPUT_PREFIX}predictions_2026.parquet"
preds_2026.to_parquet(s3_path, index=False)
print(f"Saved to S3: {s3_path} ({preds_2026.shape})")

Saved to S3: s3://march-machine-learning-mania-2026/07_submission/predictions_2026.parquet ((132133, 8))


## 7. Summary

In [15]:
print("=" * 60)
print("SUBMISSION SUMMARY")
print("=" * 60)

print(f"\nStage 1 (2022-2025 validation):")
print(f"  Rows: {len(submission_stage1):,}")
print(f"  Pred range: [{submission_stage1['Pred'].min():.3f}, {submission_stage1['Pred'].max():.3f}]")

print(f"\nStage 2 (2026 submission):")
print(f"  Rows: {len(submission_stage2):,}")
print(f"  Pred range: [{submission_stage2['Pred'].min():.3f}, {submission_stage2['Pred'].max():.3f}]")

print(f"\nMen's predictions: Ensemble (XGBoost + LightGBM + CatBoost + PyTorch)")
print(f"Women's predictions: Ensemble (XGBoost + LightGBM + CatBoost + PyTorch)")

SUBMISSION SUMMARY

Stage 1 (2022-2025 validation):
  Rows: 519,144
  Pred range: [0.020, 0.980]

Stage 2 (2026 submission):
  Rows: 132,133
  Pred range: [0.020, 0.980]

Men's predictions: Ensemble (XGBoost + LightGBM + CatBoost + PyTorch)
Women's predictions: Ensemble (XGBoost + LightGBM + CatBoost + PyTorch)
